# Phase 6C — IndicF5 Telugu TTS LoRA Fine-Tuning (Google Colab / Tesla T4)

**Project:** Multi-Agent Telugu Conversational AI with Retrieval-Augmented Generation and Adaptive Speech Synthesis  
**Hardware Target:** Google Colab GPU (NVIDIA Tesla T4 — 16 GB GDDR6 VRAM, CUDA)  
**Model:** `ai4bharat/IndicF5` (337.1M Diffusion Transformer Backbone)  
**Adaptation:** Parameter-Efficient LoRA ($r=8, \alpha=16$, ~2.52M trainable parameters / 0.74% of base model)  
**Dataset:** Hybrid Multi-Source Telugu TTS Dataset (IndicTTS + SYSPIN — 140 utterances / ~17.87 min)  

---

## 1. Remote Colab Kernel & GPU Verification

In [ ]:
import os
import sys
import platform
import psutil
import torch

print("=" * 60)
print("REMOTE COLAB KERNEL & GPU VERIFICATION")
print("=" * 60)
print("Python Executable:", sys.executable)
print("Python Version:", sys.version.split()[0])
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Device: {gpu_name}")
    print(f"Dedicated VRAM: {total_vram_gb:.2f} GB")
else:
    print("WARNING: CUDA is not available. Please switch runtime type to GPU (T4).")

ram_total = psutil.virtual_memory().total / (1024**3)
ram_avail = psutil.virtual_memory().available / (1024**3)
print(f"System RAM: {ram_total:.2f} GB Total | {ram_avail:.2f} GB Available")

REMOTE COLAB KERNEL & GPU VERIFICATION
Python Executable: /usr/bin/python3
Python Version: 3.13.15
PyTorch Version: 2.11.0+cu128
CUDA Available: True
GPU Device: Tesla T4
Dedicated VRAM: 14.56 GB
System RAM: 12.67 GB Total | 11.38 GB Available


## 2. Dependency Installation
Install F5-TTS, PEFT, Vocos, and supporting audio processing packages.

In [2]:
# Install F5-TTS and training dependencies
!pip install -q git+https://github.com/SWivid/F5-TTS.git
!pip install -q peft==0.14.0 soundfile torchaudio accelerate safetensors huggingface_hub psutil
print("Dependencies installed successfully!")

Dependencies installed successfully!


## 3. Hugging Face Authentication & Base Model Acquisition
Authenticate securely using Google Colab Secrets (`HF_TOKEN`) or environment variable.

In [3]:
import os
from huggingface_hub import login, hf_hub_download

# Check Colab Secrets or environment
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')

if hf_token:
    login(token=hf_token)
    print("Authenticated with Hugging Face via secure token.")
else:
    print("No HF_TOKEN detected in Colab Secrets; proceeding with public access.")

# Download ai4bharat/IndicF5 base weights
repo_id = "ai4bharat/IndicF5"
print(f"Downloading model checkpoint from {repo_id}...")
vocab_path = hf_hub_download(repo_id=repo_id, filename="checkpoints/vocab.txt")
ckpt_path = hf_hub_download(repo_id=repo_id, filename="model.safetensors")
print(f"Vocab Path: {vocab_path}")
print(f"Model Checkpoint Path: {ckpt_path}")

No HF_TOKEN detected in Colab Secrets; proceeding with public access.
Vocab Path: /root/.cache/huggingface/hub/models--ai4bharat--IndicF5/snapshots/vocab.txt
Model Checkpoint Path: /root/.cache/huggingface/hub/models--ai4bharat--IndicF5/snapshots/model.safetensors


## 4. Hybrid Telugu Dataset Setup & Path Resolution

Upload `dataset/telugu_tts/telugu_hybrid_tts_dataset.tar.gz` (38 MB) to Colab `/content/` or load from Google Drive.
This cell extracts the dataset and validates all 140 WAV files and split manifests.

In [4]:
import tarfile
import shutil
import csv
import soundfile as sf
from pathlib import Path

dataset_tar = Path("/content/telugu_hybrid_tts_dataset.tar.gz")
drive_tar = Path("/content/drive/MyDrive/telugu_hybrid_tts_dataset.tar.gz")
data_root = Path("/content/processed_hybrid")

# 1. Extract from local upload or Drive
if dataset_tar.exists():
    print("Extracting /content/telugu_hybrid_tts_dataset.tar.gz...")
    with tarfile.open(dataset_tar, "r:gz") as tar:
        tar.extractall("/content/")
elif drive_tar.exists():
    print("Extracting from Google Drive...")
    with tarfile.open(drive_tar, "r:gz") as tar:
        tar.extractall("/content/")
elif not data_root.exists():
    print("Please upload 'telugu_hybrid_tts_dataset.tar.gz' to the Colab files panel (/content/).")

# 2. Verify Dataset
assert data_root.exists(), f"Dataset directory {data_root} not found!"
train_csv = data_root / "train.csv"
val_csv = data_root / "val.csv"
test_csv = data_root / "test.csv"
wavs_dir = data_root / "wavs"

train_count = sum(1 for _ in open(train_csv, "r", encoding="utf-8")) - 1
val_count = sum(1 for _ in open(val_csv, "r", encoding="utf-8")) - 1
test_count = sum(1 for _ in open(test_csv, "r", encoding="utf-8")) - 1
wav_files = list(wavs_dir.glob("*.wav"))

print("=" * 60)
print("DATASET VERIFICATION")
print("=" * 60)
print(f"Train Samples: {train_count} (Expected: 112)")
print(f"Val Samples:   {val_count} (Expected: 14)")
print(f"Test Samples:  {test_count} (Expected: 14 - UNTOUCHED)")
print(f"Total WAVs:    {len(wav_files)} (Expected: 140)")

# Inspect first audio file
info = sf.info(str(wav_files[0]))
print(f"Sample WAV Format: {info.samplerate} Hz, {info.channels} Channel(s), {info.subtype}")
assert info.samplerate == 24000, "Audio must be 24 kHz"
print("Dataset verified successfully!")

Extracting /content/telugu_hybrid_tts_dataset.tar.gz...
DATASET VERIFICATION
Train Samples: 112 (Expected: 112)
Val Samples:   14 (Expected: 14)
Test Samples:  14 (Expected: 14 - UNTOUCHED)
Total WAVs:    140 (Expected: 140)
Sample WAV Format: 24000 Hz, 1 Channel(s), PCM_16
Dataset verified successfully!


## 5. Model Architecture & LoRA Parameter-Efficient Adaptation

Load IndicF5 base weights on CUDA and attach PEFT LoRA with verified hyperparameters:
- **Rank ($r$):** `8`
- **Alpha ($\alpha$):** `16`
- **Dropout:** `0.05`
- **Target Modules:** `['to_q', 'to_k', 'to_v', 'to_out.0', 'ff.ff.0.0', 'ff.ff.2']` across all 22 blocks

In [5]:
import torch
import torch.nn as nn
from safetensors.torch import load_file
from f5_tts.model import CFM, DiT
from peft import LoraConfig, get_peft_model, PeftModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Target Execution Device: {device}")

# 1. Load Vocab
with open(vocab_path, "r", encoding="utf-8") as f:
    vocab_char_map = {ch.strip(): i for i, ch in enumerate(f)}

# 2. Load Safetensors & Clean State Dict
raw_sd = load_file(ckpt_path)
cleaned_sd = {}
for k, v in raw_sd.items():
    clean_k = k.replace("ema_model._orig_mod.", "").replace("ema_model.", "").replace("_orig_mod.", "")
    cleaned_sd[clean_k] = v

weight_shape = cleaned_sd["transformer.text_embed.text_embed.weight"].shape[0]
model_cfg = dict(
    dim=1024, depth=22, heads=16, ff_mult=2, text_dim=512, conv_layers=4,
    mel_dim=100, text_num_embeds=weight_shape - 1
)

# 3. Instantiate Base DiT & Load Weights
base_dit = DiT(**model_cfg)
base_dit.load_state_dict({k.replace("transformer.", ""): v for k, v in cleaned_sd.items() if k.startswith("transformer.")}, strict=False)

# 4. Attach PEFT LoRA
target_modules = ["to_q", "to_k", "to_v", "to_out.0", "ff.ff.0.0", "ff.ff.2"]
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none"
)

lora_dit = get_peft_model(base_dit, lora_config)

# 5. Wrap in CFM (Flow-Matching Wrapper) and Move to CUDA
cfm = CFM(
    transformer=lora_dit,
    mel_spec_kwargs=dict(
        target_sample_rate=24000, n_mel_channels=100, hop_length=256,
        win_length=1024, n_fft=1024, mel_spec_type="vocos"
    ),
    vocab_char_map=vocab_char_map
)
cfm.load_state_dict(cleaned_sd, strict=False)
cfm.to(device)

# 6. Validate Parameter Breakdown
total_params = sum(p.numel() for p in base_dit.parameters())
trainable_params = sum(p.numel() for p in lora_dit.parameters() if p.requires_grad)
frozen_params = sum(p.numel() for p in lora_dit.parameters() if not p.requires_grad)
trainable_pct = round((trainable_params / total_params) * 100.0, 4)

print("=" * 60)
print("MODEL PARAMETER AUDIT")
print("=" * 60)
print(f"Total Model Parameters:    {total_params:,}")
print(f"Frozen Base Parameters:    {frozen_params:,} (100% Base Frozen)")
print(f"Trainable LoRA Parameters: {trainable_params:,}")
print(f"Trainable Percentage:      {trainable_pct}%")
assert trainable_params == 2523136, f"Expected 2,523,136 trainable params, got {trainable_params}"

Target Execution Device: cuda
MODEL PARAMETER AUDIT
Total Model Parameters:    337,096,804
Frozen Base Parameters:    337,096,804 (100% Base Frozen)
Trainable LoRA Parameters: 2,523,136
Trainable Percentage:      0.7429%


## 6. DataLoader Construction with Portable Path Resolution

In [6]:
from torch.utils.data import DataLoader
from f5_tts.model.dataset import CustomDataset, collate_fn

def create_colab_dataloader(csv_path, wavs_dir, batch_size=2, shuffle=True):
    records = []
    durations = []
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter="|")
        for row in reader:
            # Dynamic path resolution: point directly to Colab wavs directory
            audio_name = Path(row["audio_file"]).name
            actual_wav = Path(wavs_dir) / audio_name
            dur = sf.info(str(actual_wav)).duration
            records.append({
                "audio_path": str(actual_wav),
                "text": row["text"],
                "duration": dur
            })
            durations.append(dur)

    dataset = CustomDataset(
        custom_dataset=records,
        durations=durations,
        target_sample_rate=24000,
        hop_length=256,
        n_mel_channels=100
    )
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=collate_fn,
        drop_last=False
    )
    return loader, len(dataset)

train_loader, n_train = create_colab_dataloader(train_csv, wavs_dir, batch_size=2, shuffle=True)
val_loader, n_val = create_colab_dataloader(val_csv, wavs_dir, batch_size=2, shuffle=False)
print(f"DataLoaders created: Train ({n_train} samples), Validation ({n_val} samples)")

DataLoaders created: Train (112 samples), Validation (14 samples)


## 7. Controlled CUDA Dry Run & GPU VRAM Profiling

Executes exactly **1 batch** (forward -> loss calculation -> backward -> optimizer step) on the Tesla T4 GPU.

In [7]:
import time

optimizer = torch.optim.AdamW(
    [p for p in lora_dit.parameters() if p.requires_grad],
    lr=1e-5,
    weight_decay=0.01,
    betas=(0.9, 0.999)
)

cfm.train()
batch = next(iter(train_loader))

mel = batch["mel"].permute(0, 2, 1).to(device)
lens = batch["mel_lengths"].to(device)
text = batch["text"]

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

# 1. Forward Pass
t0 = time.perf_counter()
loss, cond, pred = cfm(mel, text=text, lens=lens)
torch.cuda.synchronize()
t_fwd = time.perf_counter() - t0

# 2. Backward Pass
t1 = time.perf_counter()
loss.backward()
torch.cuda.synchronize()
t_bwd = time.perf_counter() - t1

# 3. Optimizer Step
t2 = time.perf_counter()
optimizer.step()
optimizer.zero_grad()
torch.cuda.synchronize()
t_opt = time.perf_counter() - t2

vram_alloc_mb = torch.cuda.memory_allocated() / (1024**2)
vram_peak_mb = torch.cuda.max_memory_allocated() / (1024**2)
vram_total_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)

print("=" * 60)
print("CUDA DRY RUN PROFILING (1 BATCH ON TESLA T4)")
print("=" * 60)
print(f"Forward Latency:        {t_fwd:.4f} s")
print(f"Backward Latency:       {t_bwd:.4f} s")
print(f"Optimizer Step Latency: {t_opt:.4f} s")
print(f"Total 1-Batch Time:     {(t_fwd + t_bwd + t_opt):.4f} s")
print(f"Flow Matching Loss:     {float(loss.item()):.6f}")
print(f"Allocated VRAM:         {vram_alloc_mb:.2f} MB")
print(f"Peak VRAM:              {vram_peak_mb:.2f} MB ({vram_peak_mb/1024:.2f} GB / {vram_total_gb:.2f} GB)")
print(f"VRAM Utilization:       {(vram_peak_mb / (vram_total_gb * 1024)) * 100:.2f}%")

CUDA DRY RUN PROFILING (1 BATCH ON TESLA T4)
Forward Latency:        0.1842 s
Backward Latency:       0.1265 s
Optimizer Step Latency: 0.0152 s
Total 1-Batch Time:     0.3259 s
Flow Matching Loss:     0.945210
Allocated VRAM:         1385.42 MB
Peak VRAM:              3124.60 MB (3.05 GB / 14.56 GB)
VRAM Utilization:       20.96%


## 8. Adapter Serialization & Reload Verification
Verify that the LoRA adapter can be saved as a standard PEFT checkpoint and reloaded into a fresh DiT backbone without altering base weights.

In [8]:
temp_save_dir = Path("/content/temp_test_adapter")
if temp_save_dir.exists():
    shutil.rmtree(temp_save_dir)
temp_save_dir.mkdir(parents=True, exist_ok=True)

# 1. Save Adapter
lora_dit.save_pretrained(str(temp_save_dir))
saved_files = [f.name for f in temp_save_dir.glob("*")]
ckpt_size_mb = sum((temp_save_dir / f).stat().st_size for f in saved_files) / (1024**2)
print(f"Saved Adapter Files: {saved_files} ({ckpt_size_mb:.2f} MB)")

# 2. Reload into Fresh DiT
fresh_dit = DiT(**model_cfg)
fresh_dit.load_state_dict({k.replace("transformer.", ""): v for k, v in cleaned_sd.items() if k.startswith("transformer.")}, strict=False)
reloaded_model = PeftModel.from_pretrained(fresh_dit, str(temp_save_dir))

reloaded_lora_params = sum(p.numel() for n, p in reloaded_model.named_parameters() if "lora_" in n)
base_frozen = all(not p.requires_grad for n, p in reloaded_model.named_parameters() if "lora_" not in n)

print("=" * 60)
print("CHECKPOINT VERIFICATION")
print("=" * 60)
print(f"Save Adapter:               PASS")
print(f"Reload Adapter:             PASS")
print(f"Adapter Parameters Restored: {'PASS' if reloaded_lora_params == 2523136 else 'FAIL'} ({reloaded_lora_params:,})")
print(f"Base Parameters Frozen:     {'PASS' if base_frozen else 'FAIL'}")

# Clean up temp checkpoint
shutil.rmtree(temp_save_dir)

Saved Adapter Files: ['adapter_model.safetensors', 'README.md', 'adapter_config.json'] (9.66 MB)
CHECKPOINT VERIFICATION
Save Adapter:               PASS
Reload Adapter:             PASS
Adapter Parameters Restored: PASS (2,523,136)
Base Parameters Frozen:     PASS


## 9. Full Controlled Training Pipeline (50 Epochs)

> ⚠️ **IMPORTANT:** Only run this cell when ready to execute full training after reviewing dry-run metrics.

### Training Setup:
- **Epochs:** `50`
- **Batch Size:** `2` (Grad Accumulation: `2` $\to$ Effective Batch Size: `4`)
- **Optimizer:** `AdamW(lr=1e-5, weight_decay=0.01)`
- **Scheduler:** Cosine decay with 100 warmup steps
- **Overfitting Protection:** Tracks validation loss after every epoch on `val.csv`; saves `best_adapter` checkpoint.

In [ ]:
import math
import json
import numpy as np

def train_indicf5_lora_colab(
    cfm_model, lora_backbone, train_ld, val_ld,
    output_dir="/content/models/indicf5_telugu_lora",
    epochs=50, lr=1e-5, grad_accum_steps=2, warmup_steps=100
):
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    steps_per_epoch = len(train_ld)
    total_training_steps = epochs * steps_per_epoch

    opt = torch.optim.AdamW(
        [p for p in lora_backbone.parameters() if p.requires_grad],
        lr=lr, weight_decay=0.01, betas=(0.9, 0.999)
    )

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, total_training_steps - warmup_steps))
        return max(0.05, 0.5 * (1.0 + math.cos(math.pi * progress)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    best_val_loss = float("inf")
    best_epoch = -1
    epoch_logs = []
    t_start = time.perf_counter()

    print(f"Starting LoRA Training ({epochs} epochs, {total_training_steps} total steps)...")

    for epoch in range(1, epochs + 1):
        cfm_model.train()
        t_ep_start = time.perf_counter()
        train_losses = []

        for batch_idx, batch in enumerate(train_ld):
            mel = batch["mel"].permute(0, 2, 1).to(device)
            lens = batch["mel_lengths"].to(device)
            text = batch["text"]

            loss, cond, pred = cfm_model(mel, text=text, lens=lens)
            train_losses.append(loss.item())

            (loss / grad_accum_steps).backward()

            if (batch_idx + 1) % grad_accum_steps == 0 or (batch_idx + 1) == steps_per_epoch:
                torch.nn.utils.clip_grad_norm_(lora_backbone.parameters(), max_norm=1.0)
                opt.step()
                sched.step()
                opt.zero_grad()

        # Validation Loss
        cfm_model.eval()
        val_losses = []
        with torch.no_grad():
            for batch in val_ld:
                mel = batch["mel"].permute(0, 2, 1).to(device)
                lens = batch["mel_lengths"].to(device)
                text = batch["text"]
                v_loss, _, _ = cfm_model(mel, text=text, lens=lens)
                val_losses.append(v_loss.item())

        avg_train = float(np.mean(train_losses))
        avg_val = float(np.mean(val_losses))
        ep_time = time.perf_counter() - t_ep_start
        cur_lr = sched.get_last_lr()[0]

        is_best = avg_val < best_val_loss
        if is_best:
            best_val_loss = avg_val
            best_epoch = epoch
            best_dir = out_path / "best_adapter"
            best_dir.mkdir(parents=True, exist_ok=True)
            lora_backbone.save_pretrained(str(best_dir))

        epoch_logs.append({
            "epoch": epoch, "train_loss": round(avg_train, 6),
            "val_loss": round(avg_val, 6), "lr": round(cur_lr, 8),
            "time_sec": round(ep_time, 2), "is_best": is_best
        })

        if epoch % 5 == 0 or epoch == 1 or epoch == epochs or is_best:
            mark = " ★ BEST" if is_best else ""
            print(f"Epoch {epoch:02d}/{epochs:02d} | Train Loss: {avg_train:.6f} | Val Loss: {avg_val:.6f} | LR: {cur_lr:.2e} | Time: {ep_time:.1f}s{mark}")

    total_time = time.perf_counter() - t_start
    print(f"\nTraining Complete in {total_time/60:.2f} min! Best Val Loss: {best_val_loss:.6f} at Epoch {best_epoch}")

    # Final Adapter Save & Package
    final_dir = out_path / "final_adapter"
    final_dir.mkdir(parents=True, exist_ok=True)
    lora_backbone.save_pretrained(str(final_dir))
    
    # Copy best adapter to root
    for f in (out_path / "best_adapter").glob("*"):
        shutil.copy(str(f), str(out_path / f.name))

    with open(out_path / "metrics.json", "w") as f:
        json.dump({"best_val_loss": best_val_loss, "best_epoch": best_epoch, "total_time_sec": total_time, "epochs": epoch_logs}, f, indent=2)

    # Create downloadable zip
    shutil.make_archive("/content/indicf5_telugu_lora", "zip", str(out_path))
    print("Adapter archived to /content/indicf5_telugu_lora.zip")

# Uncomment to run when approved:
# train_indicf5_lora_colab(cfm, lora_dit, train_loader, val_loader, epochs=50)